# Recurrent Neural Networks: Forecasting Photovoltaic Generation

- The dataset file can be found at https://datadryad.org/dataset/doi:10.5061/dryad.m37pvmd99
- In the 'Library & System Setup' block, there is a line to set the root path of a locally downloaded dataset file from the above link. The file extraction should happen automatically after this file path is changed, and no other file paths should need to be changed.
  
- No changes are needed for the data format from the original download except for in the metadata file, where an error was found, which was corrected locally. However, this does not matter for the project since this model only predicts for a single PV site's data, and does not generalise to all PV sites or use the metadata file.
- The code ran faster locally (MacOS) with a CPU, rather than a GPU (MPS). However, there is a commented-out line for running code on the GPU if required (Windows OS w/ CUDA).
- Additional libraries past PyTorch were installed for this project (Optuna, SHAP); however, they are not essential to the main code running, and have been commented out since they were either not implemented or take far too long to run.

## Library & System Setup

In [ ]:
#General Libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import time

#Machine Learning Libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn import metrics
from sklearn.decomposition import PCA

#PyTorch Libraries
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Subset, ConcatDataset
from torch.profiler import profile, record_function, ProfilerActivity

#Further analysis and optimisation libraries
# import optuna
# import shap

# Libraries for data extraction of PV metadata (not crucial, so can be left out, but produces a graph as to why this was abandoned)
from rdflib import Graph
import datetime

print('VALID: Libaries imported.')

In [ ]:
#NOTE: Change path to dataset folder on the local machine
dataset_root = '/Users/Hayden/Desktop/University/Year 3/Advanced Computational/A2 - Solar Forecast/Dataset'

if not os.path.exists(dataset_root):
    print('ERROR: Dataset root path does not exists. Check file path is correct or if files are downloaded locally.')
else:
    print('VALID: Dataset root path exists.')

In [ ]:
# device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
# device = torch.device('cuda') if torch.backends.cuda.is_available() else torch.device('cpu')
device = torch.device('cpu')

print(f'Device: {device}')

# Single PV Station Forecast (SQ567)

## Data Preparation 

In [ ]:
#Function for joining meteorological data files within dataset
def met_files_merge(root):
    folders = [f for f in os.listdir(root) if os.path.isdir(os.path.join(root, f))] #Array of key met data folders
    
    df = None
    
    for folder in folders: #Loop over each data folder within the entire data folder
        folder_path = os.path.join(met_root, folder) #Find the current data's folder path

        #AI Code / ChatGPT / 'How can I assign an array from a list of files, which should only contain CSV or Excel files?'
        files = [f for f in os.listdir(folder_path) if f.endswith('.csv') or f.endswith('.xlsx')] #Array of files ending in csv and xlsx within current weather data folder
        
        df_files = [] #Create empty temporary array to concat data by year into single column
        
        for file in files: #Loop over files within current data folder
            
            file_path = os.path.join(folder_path, file) #Find exact file path
            
            if file.endswith('.csv'):
                df_file = pd.read_csv(file_path) #Read csv if ends as such
            elif file.endswith('.xlsx'):
                df_file = pd.read_excel(file_path) #Read excel file is ends as such
                
            df_files.append(df_file) #Add that year of weather data to temp dataframe
            
        df_folder = pd.concat(df_files, ignore_index=True) #For weather data of same type just add on to one another
        
        if df is None: #If this is the first data folder being collected, then this will run
            df = df_folder
        else:
            df = pd.merge(df, df_folder, on='Time', how='outer') #If it's not the first folder, simply merge the main met dataframe with the temp weather variable dataframe

    df['Time'] = pd.to_datetime(df['Time']) #Set time column to correct time series format
    df = df.set_index('Time').sort_index() #Set dataframe index to time and sort
    
    return df

### Meteorological Dataset

In [ ]:
#Met data
met_root = os.path.join(dataset_root, 'Time series dataset/Meteorological dataset') #Complete file path to meterological data folder
df_met = met_files_merge(met_root) #Create met dataframe

#AI Code / ChatGPT / 'Given two dataframes that are time-series in nature but have different resolutions of time, how can you resample one to the other to align their rows for the merge?'
df_met = df_met.resample('15min').mean().interpolate(method='time') #Change time resolution to every 15mins to align with PV stations measurement resolution, interpolate any missing cells

#Create cyclical time columns for better training
df_met['hour'] = df_met.index.hour + df_met.index.minute / 60
df_met['day'] = df_met.index.dayofyear
# df_met['month'] = df_met.index.month

#Make sinusoidal
df_met['hour_sin'] = np.sin(2 * np.pi * df_met['hour'] / 24)
df_met['hour_cos'] = np.cos(2 * np.pi * df_met['hour'] / 24)

df_met['day_sin'] = np.sin(2 * np.pi * df_met['day'] / 365) #Was debated as 365.25 but left as 365 since no leap years in dataset
df_met['day_cos'] = np.cos(2 * np.pi * df_met['day'] / 365)

# df_met['month_sin'] = np.sin(2*np.pi*df_met['month']/12) #Removed as seemed to mess up model for some reason
# df_met['month_cos'] = np.cos(2*np.pi*df_met['month']/12)

df_met = df_met.drop(['hour', 'day'], axis=1) #Drop non-cyclical time columns

feature_cols = list(df_met.columns) #get generic name of feature columns for future use

title = 'MISSING DATA CHECK'
print('\n' + '='*len(title))
print(title)
print('='*len(title))
print(df_met.isna().sum())

### Single PV Generation Dataset

In [ ]:
#Defines daata for single PV site we attempt to forecast: SQ567
PV_path = 'Time series dataset/PV generation dataset/PV stations with panel level optimizer/Site level dataset/SQ567.csv'

file_path = os.path.join(dataset_root, PV_path)

df_PV = pd.read_csv(file_path)
df_PV['Time'] = pd.to_datetime(df_PV['Time'])

df_PV = df_PV.set_index('Time').sort_index().interpolate(method='time') #Fill in missing indexes

title = 'MISSING DATA CHECK'
print('\n' + '='*len(title))
print(title)
print('='*len(title))
print(df_PV.isna().sum())

target_cols = list(df_PV.columns) #get generic name of target columns for future use

### Merged Datasets

In [ ]:
#Merge met and PV data
df = pd.merge(df_met, df_PV, on='Time') #Join together weather data and target PV dataframes

title = 'MISSING DATA CHECK'
print('\n' + '='*len(title))
print(title)
print('='*len(title))
print(df.isna().sum())

df.head()

### Dataset Analysis

In [ ]:
#Plot merged dataset
n_cols = 2
n_rows = int(round(df.shape[1] / n_cols, 0)) #Calculate number of rows from number of features and columns

fig, ax = plt.subplots(nrows=n_rows, ncols=n_cols, sharex=True, figsize=(12, 8)) #Plot together and share axes
for i, ax in enumerate(fig.axes):
    sns.lineplot(data=df.iloc[:, i], ax=ax) #Plot lineplot for each column in a new axes
    ax.tick_params(axis='x', rotation=30, labelsize=10, length=0)
    ax.grid(True)
fig.tight_layout()
plt.savefig('Plots/Analysis/dataset.pdf')
plt.show()

### Transform & Seperate Data

In [ ]:
scalar_X = MinMaxScaler() #Used for PV energy daat which normalises data between 0 and 1 (since we physically cannot generate negative energy)
scalar_y = MinMaxScaler() #Same as above but for output data instead of input data

def data_prep(df, seq_len, batch_size, plot=False):
    #Get index of seperation of datasets
    train_idx_end = int(0.7 * len(df))
    val_idx_end = int(0.85 * len(df))
    test_idx_end = int(len(df))
    
    #Transform separated datasets
    X_train = scalar_X.fit_transform(df[X_columns].iloc[:train_idx_end].to_numpy())
    y_train = scalar_y.fit_transform(df[y_columns].iloc[:train_idx_end].to_numpy())

    #Does not fit scalar to val or test to prevent potential data leakage
    X_val = scalar_X.transform(df[X_columns].iloc[train_idx_end:val_idx_end].to_numpy()) 
    y_val = scalar_y.transform(df[y_columns].iloc[train_idx_end:val_idx_end].to_numpy())
    
    X_test = scalar_X.transform(df[X_columns].iloc[val_idx_end:test_idx_end].to_numpy())
    y_test = scalar_y.transform(df[y_columns].iloc[val_idx_end:test_idx_end].to_numpy())
    
    #Concatenate all of the transformed files and store as a tensor
    X = torch.Tensor(np.concatenate([X_train, X_val, X_test])).to(device)
    y = torch.Tensor(np.concatenate([y_train, y_val, y_test])).to(device)
    data = X, y #Store as single object for easier returning
    
    #Create dataset sequence from X, y, sequence length information
    dataset = Sequence(X, y, seq_len) #Dataset looks back at the last 96 indexes of 15min intervals to predict next value

    
    #AI Code / ChatGPT / 'Since introducing my sequence to my dataset tensors, there is a missmatch in dimensions, why is this?'
    train_idx_end = train_idx_end - seq_len #Find final index positions of separated datasets, accounting for the minimum dataset size due to sequence length
    val_idx_end = val_idx_end - seq_len
    test_idx_end = len(dataset)
    
    #Create data subsets for training and validation
    train_data = Subset(dataset, range(0, train_idx_end))
    val_data = Subset(dataset, range(train_idx_end, val_idx_end))
    test_data = Subset(dataset, range(val_idx_end, test_idx_end))

    #Create loaders for easy data inputting for the model
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False) #Shuffle off to keep chronological orderin for training 
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    
    if plot is True:
        #Plot separation of the dataset into train, validation, and test regions
        n_cols = 2
        n_rows = int(round(df.shape[1] / n_cols, 0))
        
        fig, ax = plt.subplots(nrows=n_rows, ncols=n_cols, sharex=True, figsize=(12, 8))
        for i, ax in enumerate(fig.axes):
            sns.lineplot(data=df.iloc[:train_idx_end, i], ax=ax)
            sns.lineplot(data=df.iloc[train_idx_end:val_idx_end, i], ax=ax)
            sns.lineplot(data=df.iloc[val_idx_end:test_idx_end, i], ax=ax)
            ax.tick_params(axis='x', rotation=30, labelsize=10, length=0)
            ax.grid(True)
        fig.tight_layout()
        plt.savefig('Plots/Training/dataset.pdf')
        plt.show()
    
    return train_loader, val_loader, test_loader, data, scalar_X, scalar_y

### Sequence Object

In [ ]:
#Sequence of time window to use in order to predict future values
class Sequence(torch.utils.data.Dataset):
    def __init__(self, X, y, seq_len):
        self.X = X
        self.y = y
        self.seq_len = seq_len

    def __len__(self):
        n_seq = len(self.X) - self.seq_len #Number of sequences that can fit in the dataset is the size of the dataset subtract one full sequence, since the minimum dataset size is one full sequence (can't have half a sequence)
        
        return n_seq

    def __getitem__(self, idx):
        X_seq = self.X[idx : idx + self.seq_len] #Training data in current sequence
        y_target = self.y[idx + self.seq_len] #Target data at end of sequence
        
        return X_seq, y_target

## Model Classes

### GRU

In [ ]:
class ModelGRU(nn.Module): #Custom python nn that inherits nn.Module class from PyTorch 
    def __init__(self, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, dropout_val):
        super().__init__() #Initalise child 'GRU' model methods, while inheriting nn.Module methods
        self.loss_function = loss_function
        self.optimiser_function = optimiser_function
        self.hidden_size = hidden_size #Number of hidden states within each neural layer
        self.num_layers = num_layers #Number of stacked layers of neurons withon model
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_val) #GRU method from PyTorch libary
        self.fc = nn.Linear(hidden_size, output_size) #Fully connected layer transforms dense layers into output layers
        
    def forward(self, X):
        h0 = torch.zeros(self.num_layers, X.size(0), self.hidden_size).to(X.device) #Initial hidden state all zeros to ensure predictions not from previous inputs sequence
        
        out, _ = self.gru(X, h0) #Output for all timesteps within data x
        out = out[:, -1, :] #Output at last hidden
        
        out = self.fc(out) #Fully connected output at final tempstep
        
        return out
    
    def training_step(self, batch):
        X, y = batch
        y_hat = self(X) #Predicts target data based on input

        if self.loss_function == 'L1':
            loss = nn.functional.l1_loss(y_hat, y) #Compute L1 loss (MAE)
        elif self.loss_function == 'L2':
            loss = nn.functional.mse_loss(y_hat, y) #Compute L2 loss (MSE)
        elif self.loss_function == 'Huber':
            loss = nn.functional.huber_loss(y_hat, y) #Compute Hubber loss
        else:
            print('ERROR: Invalid loss function input.')

        #Compute r2 score for training step
        r2 = metrics.r2_score(y.detach().numpy(), y_hat.detach().numpy()) #Some AI Code / ChatGPT / 'I am attempting to calculate the r2 score from a target and predicted value in a GRU, however, the object types are not cooperating, why might this be?'
    
        return loss, r2 #Returns for backpropagation

### LSTM

In [ ]:
class ModelLSTM(nn.Module):
    def __init__(self, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, dropout_val):
        super(ModelLSTM, self).__init__()
        self.loss_function = loss_function
        self.optimiser_function = optimiser_function
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_val)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        h0 = torch.zeros(self.num_layers, X.size(0), self.hidden_size).to(X.device)
        c0 = torch.zeros(self.num_layers, X.size(0), self.hidden_size).to(X.device) #Additional cell state which goes alongside hidden state in LSTM

        out, (hn, cn) = self.lstm(X, (h0, c0))
        out = out[:, -1, :]
        
        out = self.fc(out)

        return out

    def training_step(self, batch):
        X, y = batch
        y_hat = self(X)

        if self.loss_function == 'L1':
            loss = nn.functional.l1_loss(y_hat, y) #Compute L1 loss (MAE)
        elif self.loss_function == 'L2':
            loss = nn.functional.mse_loss(y_hat, y) #Compute L2 loss (MSE)
        elif self.loss_function == 'Huber':
            loss = nn.functional.huber_loss(y_hat, y) #Compute Hubber loss
        else:
            print('ERROR: Invalid loss function input.')

        r2 = metrics.r2_score(y.detach().numpy(), y_hat.detach().numpy()) #Compute r2 score for training step

        return loss, r2

### Simple RNN

In [ ]:
class ModelRNN(nn.Module):
    def __init__(self, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, dropout_val):
        super(ModelRNN, self).__init__()
        self.loss_function = loss_function
        self.optimiser_function = optimiser_function
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, nonlinearity='relu', dropout=dropout_val) #ReLU is a function applied on hidden states to learn non-linear behaviour
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        h0 = torch.zeros(self.num_layers, X.size(0), self.hidden_size).to(X.device)

        out, hn = self.rnn(X, h0)
        out = out[:, -1, :]
        
        out = self.fc(out)

        return out

    def training_step(self, batch):
        X, y = batch
        y_hat = self(X)

        if self.loss_function == 'L1':
            loss = nn.functional.l1_loss(y_hat, y) #Compute L1 loss (MAE)
        elif self.loss_function == 'L2':
            loss = nn.functional.mse_loss(y_hat, y) #Compute L2 loss (MSE)
        elif self.loss_function == 'Huber':
            loss = nn.functional.huber_loss(y_hat, y) #Compute Hubber loss
        else:
            print('ERROR: Invalid loss function input.')

        r2 = metrics.r2_score(y.detach().numpy(), y_hat.detach().numpy()) #Compute r2 score for training step

        return loss, r2

## Trainer Classes & Functions

In [ ]:
class Trainer():
    def __init__(self, model, num_epochs, learning_rate, train_loader, val_loader, early_stopper):
        self.model = model
        self.num_epochs = num_epochs
        
        self.loss_train = np.zeros(num_epochs)
        self.loss_val = np.zeros(num_epochs)

        self.r2_train = np.zeros(num_epochs)
        self.r2_val = np.zeros(num_epochs)
        
        self.train_loader = train_loader
        self.val_loader = val_loader

        self.early_stopper = early_stopper

        # Optimisers for loss optimisation - choose between Adam and SGD
        if self.model.optimiser_function == 'Adam':
            self.optimiser = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        elif self.model.optimiser_function == 'SGD':
            self.optimiser = torch.optim.SGD(self.model.parameters(), lr=learning_rate)
        else:
            print('ERROR: Invalid optimiser function input.')

    def fit(self, trial=None): #Trial left as an artefact from optuna hyperparam tuning
        total_time = 0
        for epoch in range(self.num_epochs):
            
            start_time = time.time()
            
            #Training dataset
            self.model.train() #Switches model to training mode

            #Set loss and r2 for training to 0 for epoch
            loss_train = []
            r2_train = []
            
            for batch in self.train_loader: #Seperate training via batches
                loss, r2 = self.model.training_step(batch) #Calculate loss from training dataset batch
                loss.backward() #Calculates the gradient of the current loss position, how much each parameter is contributing, and in what direction the model needs to move to reduce the loss
                self.optimiser.step() #Update weights of params from grad of loss
                self.optimiser.zero_grad() #Reset grad for next calculation
                loss_train.append(loss.item()) #Append training loss from batch
                r2_train.append(r2) #Append r2 loss from batch
                
            self.loss_train[epoch] = np.array(loss_train).mean() #Mean loss over all batches in train loader
            self.r2_train[epoch] = np.array(r2_train).mean() #Mean r2 score over all batches in train loader
            
            #Validation dataset
            self.model.eval() #Switches model to evaluating mode (to prevent some features such as dropout)
            
            loss_val = []
            r2_val = []
            
            with torch.no_grad(): #Detaches tensor from loss grad, allowing for faster computation as less memory is used since we never call backwards
                for batch in self.val_loader:
                    loss, r2 = self.model.training_step(batch)
                    loss_val.append(loss.item())
                    r2_val.append(r2)
                
            self.loss_val[epoch] = np.array(loss_val).mean()
            self.r2_val[epoch] = np.array(r2_val).mean()
            
            end_time = time.time()
            delta_time = end_time - start_time
            total_time += delta_time

            #Output epoch training/ validation loss and r2 scores and time taken
            print(f'Epoch [{epoch+1}/{self.num_epochs}] | Train Loss: {self.loss_train[epoch]:.4f} | Validation Loss: {self.loss_val[epoch]:.4f} | Train R2: {self.r2_train[epoch]:.2f} | Validation R2: {self.r2_val[epoch]:.2f} | {delta_time/60:.2f}mins')

            #Check for early stopping
            self.early_stopper(self.loss_val[epoch], self.model)
            if self.early_stopper.early_stop:
                print(f'Early Stopping @ [[{epoch+1}/{self.num_epochs}]]')
                break
                    
            #Artefact left from hyperparameter optimisation
            if trial is not None: #If inside a trail
                trial.report(self.loss_val[epoch], epoch) #Return loss validation to trial report for specific epoch of hyperparameters
                if trial.should_prune(): #Checks if trail should be pruned to determine if the trial should continue or not
                    raise optuna.exceptions.TrialPruned() #Trail has been pruned due to poor performance and relives compuational power

        self.early_stopper.load_best_model(self.model) #Following early stopping/ end of training cycle, load best performing model
        
        print(f'\nTotal Training Time: {total_time/3600:.2f} hrs')

        #Really this should've been in the training loop to gather more accurate info in training, improvement for next time
        title = 'COMPUTATION SUMMARY'
        print('\n' + '='*len(title))
        print(title)
        print('='*len(title))

        self.model.eval() #Set model to evaluation mode

        with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof: #records 'TorchScript functions and user-defined code labels'
            with torch.no_grad():
                with record_function('model_inference'):
                    for X, y in self.train_loader:
                        X, y = X.float().to(device), y.float().to(device)
                        self.model(X)
                        break #Only probes for a single batch to save time
        
        print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=10))

In [ ]:
class EarlyStopping():
    def __init__(self, patience, delta):
        self.patience = patience #Number of epochs to wait before stopping training
        self.delta = delta #Error in which model defines actual model improvement to warrant training continuing 
        self.best_val = None #Updated each time a better validation loss is found
        self.early_stop = False
        self.counter = 0 #Increases as model performance doesn't increase
        self.best_model_state = None #Model weights at the index of training of the best validation loss

    def __call__(self, val_loss, model):
        
        if self.best_val is None: #Initial epoch
            self.best_val = val_loss
            self.best_model_state = model.state_dict()
            
        elif val_loss < self.best_val - self.delta: #Model performance is still improving
            self.best_val = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0 

        else: #Model performance no improvement
            self.counter += 1 

            if self.counter >= self.patience: #Model performance plateau if within patience, no performance increase
                self.early_stop = True

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [ ]:
def train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta):

    #Instantitae model object (either GRU or LSTM, depending on what is wanted by the user)
    if architecture == 'GRU':
        model = ModelGRU(loss_function=loss_function, optimiser_function=optimiser_function, input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, dropout_val=dropout_val).to(device)
    elif architecture == 'LSTM':
        model = ModelLSTM(loss_function=loss_function, optimiser_function=optimiser_function, input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, dropout_val=dropout_val).to(device)
    elif architecture == 'RNN':
        model = ModelRNN(loss_function=loss_function, optimiser_function=optimiser_function, input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, dropout_val=dropout_val).to(device)
    else:
        print('ERROR: Invalid model architecture input')

    early_stopper = EarlyStopping(patience, delta)
    
    #Create trainer object based on requirements of training and data loaders
    trainer = Trainer(model, num_epochs, learning_rate, train_loader, val_loader, early_stopper)
    trainer.fit()
    
    epochs = np.arange(1, trainer.num_epochs + 1)

    #Plot loss and r2
    fig, (ax1, ax2) = plt.subplots(2, figsize=(12,8))

    ax1.plot(epochs, trainer.loss_train, label='Train')
    ax1.plot(epochs, trainer.loss_val, label='Validation')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'Loss Function Results')
    ax1.grid(True)
    ax1.legend()

    ax2.plot(epochs, trainer.r2_train, label='Train')
    ax2.plot(epochs, trainer.r2_val, label='Validation')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('R2 Score')
    ax2.set_title(f'R2 Score Results')
    ax2.grid(True)
    ax2.legend()

    fig.tight_layout()
    plt.savefig(f'Plots/Training/SOLO_loss_{architecture}_{loss_function}_{optimiser_function}.pdf')
    plt.show()

    return model, trainer

## Evaluation Functions

In [ ]:
#PRECTION FUNCTION TO USE TRAINED MODEL AND PREDICT FORECAST OF SOLAR FOR HOWEVER MANY DAYS AHEAD
def prediction(model, data, scalar_y, forecast_idx, forecast_len):
    model.eval() #Set model to evaluation mode to stop dropout and batch normalisation

    X, y = data #Get X and y tensors

    lookback_idx = forecast_idx - seq_len #Find index at where to start looking at data for forecast index

    #Some AI Code / ChatGPT / 'There is a missalingment between my tensor dimension and input data, how can I resolve this, can i just create a dummy dimension?'
    X_initial_window = X[lookback_idx : forecast_idx].unsqueeze(0) #Training data within window, add additional dimension to tensor
    y_true = y[forecast_idx : forecast_idx + forecast_len] #Result from validation data used to predict data

    latency = []

    with torch.no_grad():

        X_current_window = X_initial_window.clone() #Set the first current window to the initial window clone
        y_hats = [] #Predictions array

        #Predict new value, update window, predict next value
        for i_window in range(forecast_len): 
            start_time = time.time()
            y_hat = model(X_current_window) #Prediction of targets from current window of input data
            y_hats.append(y_hat) #Add to list of predictions
            
            current_window_step = X_current_window[:, 1:, :] #Take away 1 row of data from start of window
            next_window_step = X[forecast_idx + i_window, :].unsqueeze(0).unsqueeze(0) #Find next row of data for inptu window

            X_current_window = torch.cat([current_window_step, next_window_step], dim=1) #Concat for new rollwoing window data

            final_time = time.time()
            latency.append(final_time - start_time) #Append time taken for single prediction to latency

        y_hats = torch.cat(y_hats) #Concat all of y predictions

    #Transform data back to interpretable info
    y_trues = scalar_y.inverse_transform(y_true.numpy()) 
    y_hats = scalar_y.inverse_transform(y_hats.numpy())
    
    mean_latency = np.mean(latency)
    
    #EVALUATION METRICS
    mse = metrics.mean_squared_error(y_trues, y_hats)
    rmse = np.sqrt(mse)
    mae = metrics.mean_absolute_error(y_trues, y_hats)
    r2 = metrics.r2_score(y_trues, y_hats)

    title = 'EVAULATION METRICS'
    print('=' * len(title))
    print(title)
    print('=' * len(title))
    
    print(f'\nStart Date: {start_date}')
    print(f'Num Days Forecasted: {num_days:.2f}')

    print(f'\nMean Latency: {mean_latency*1e3:.2f}ms')
    
    print(f'\nMSE: {mse:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'MAE: {mae:.2f}')
    print(f'R2 Score: {r2:.2f}')

    return y_trues, y_hats

In [ ]:
def plot_forecast(y_trues, y_hats, title, forecast_idx, forecast_len):
    main_title = f'Forecast - {title}'

    time_index = df.index[forecast_idx : forecast_idx + forecast_len] #Get time index over forecast
    
    fig, ax = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(12, 8))
    fig.suptitle(main_title)
    
    for i, ax in enumerate(fig.axes):
        target_data = pd.Series(y_trues[:,i], index=time_index)
        predictions_data = pd.Series(y_hats[:,i], index=time_index)
        
        sns.lineplot(data=target_data, label='Target', alpha=0.7, ax=ax)
        sns.lineplot(data=predictions_data, label='Predicted', alpha=0.7, ax=ax)
        ax.tick_params(axis='x', rotation=30, labelsize=10, length=0)
        ax.set_ylabel(y_columns[i])
        ax.grid(True)
    fig.tight_layout()
    plt.savefig(f'Plots/Evaluate/{main_title}.pdf')
    plt.show()


def plot_residuals(y_trues, y_hats, title, forecast_idx, forecast_len):
    residuals = y_trues - y_hats
    mean_residual = np.mean(residuals)
    std_residual = np.std(residuals)

    main_title = f'Residuals - {title}'
    
    time_index = df.index[forecast_idx : forecast_idx + forecast_len]

    #Lineplot
    fig, ax = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(12, 8))
    fig.suptitle(main_title)
    
    for i, ax in enumerate(fig.axes):
        residual_data = pd.Series(residuals[:,i], index=time_index)
        sns.lineplot(data=residual_data, alpha=0.7, ax=ax)
        ax.tick_params(axis='x', rotation=30, labelsize=10, length=0)
        ax.set_ylabel(y_columns[i])
        ax.grid(True)
    fig.tight_layout()
    plt.savefig(f'Plots/Evaluate/lineplot_{main_title}.pdf')
    plt.show()

    #Histogram
    fig, ax = plt.subplots(nrows=2, ncols=1, sharex=False, figsize=(12, 8))
    fig.suptitle(main_title)
    
    for i, ax in enumerate(fig.axes):
        residual_data = pd.Series(residuals[:,i], index=time_index)
        sns.histplot(data=residual_data, alpha=0.7, ax=ax)
        ax.tick_params(axis='x', rotation=30, labelsize=10, length=0)
        ax.set_ylabel(y_columns[i])
        ax.grid(True)
    fig.tight_layout()
    plt.savefig(f'Plots/Evaluate/histogram_{main_title}.pdf')
    plt.show()

## GRU: Main Model

This GRU Model is the main one used throughout the program and analysis within the report.

### Train Model

The following inputs can define the model:
- Architecture - 'GRU', 'LSTM', 'RNN'
- Loss Function - 'L1', 'L2', 'Huber'
- Optimiser Function - 'Adam', 'SGD'

In [ ]:
#Define input and output variables within model
X_columns = feature_cols
y_columns = target_cols

#Define the model architecture for training
architecture = 'GRU'
loss_function = 'L1'
optimiser_function = 'Adam'

#Hyperparameters
seq_len = 96 * 3 #Number of 15min indexes within the sequence training window (3 days)
batch_size = 64 #Number of training samples in a single forward and backward pass before weights are updated

input_size = len(X_columns) #Size of input vector
hidden_size = 128 #Size of hidden state vector
num_layers = 2 #Number of stacked GRUs/ recurrent layers
output_size = len(y_columns) #Size of output vector

learning_rate = 1e-4 #Step of learning between epochs
num_epochs = 50 #Number of maximum epochs to run for training

#REGULARISATION
dropout_val = 0.2 #Regulaisation technique
patience = 5 #Early stopping patience
delta = 0.001 #Early stopping delta

In [ ]:
#Gather data from data prep function
train_loader, val_loader, test_loader, data, scalar_X, scalar_y = data_prep(df, seq_len, batch_size, plot=True)

#Define model training for user output to verify the information below applied to the model they want to run
model_design = f'{architecture}, {loss_function}, {optimiser_function}'
title = f'TRAINING MODEL: {model_design}'
print('='*len(title))
print(title)
print('='*len(title))

#Train model and define model object and training object for future predictions and evaluation
model_GRU, trainer_GRU = train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta)

### Save & Load Model

If the model runtime is too long, there is a file containing the model in the cell below, which can be loaded. If the code below is wanted to run, then comment out the bottom two lines and make sure the .pth file is in the dataset folder.

In [ ]:
# path = os.path.join(dataset_root, 'modelGRU.pth')
# torch.save(model_GRU.state_dict(), path)

# model_GRU = ModelGRU(loss_function=loss_function, optimiser_function=optimiser_function, input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, dropout_val=dropout_val).to(device)
# model_GRU.load_state_dict(torch.load(path))
# model_GRU.eval()

### Evaluate Model

#### Test Dataset Results

In [ ]:
#Run prediction for entire test dataset
start_date = '2023-09-10 00:00:00' #Date at which test index started in SQ567
num_days = 107 #Number of days within test dataset

forecast_idx = df.index.get_loc(start_date)
forecast_len = int(96 * num_days)

y_trues, y_hats = prediction(model_GRU, data, scalar_y, forecast_idx, forecast_len) #Determine target and predicted values over test region

#Plot Data and Analyse
title = f'Entire Test Dataset - {model_design} '
plot_forecast(y_trues, y_hats, title, forecast_idx, forecast_len)
plot_residuals(y_trues, y_hats, title, forecast_idx, forecast_len)

#### Errorneous Irradiance Data Results

In [ ]:
num_days = 7 #Observe week 1 of test region

forecast_idx = df.index.get_loc(start_date)
forecast_len = int(96 * num_days)

#Predict forecast for dates from the model to get evaluation metrics
y_trues, y_hats = prediction(model_GRU, data, scalar_y, forecast_idx, forecast_len)

title = f'Week 1 of Test Dataset - {model_design}'
plot_forecast(y_trues, y_hats, title, forecast_idx, forecast_len)
plot_residuals(y_trues, y_hats, title, forecast_idx, forecast_len)

#Plot irradiance data over the same dates
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

fig.suptitle('Irradiance Against True Energy Generation')

#Some AI Code / ChatGPT / 'How can I gather data from the irradiance column over a range of indexes that I have stored within variables?'
ax1.plot(df.iloc[forecast_idx:forecast_idx+forecast_len].index, df.iloc[forecast_idx:forecast_idx+forecast_len]['Irradiance (W/m2)'])
ax1.set_ylabel('Irradiance (W/m2)')
ax1.tick_params(axis='x', rotation=30, labelsize=10, length=0)
ax1.grid(True)

ax2.plot(df.iloc[forecast_idx:forecast_idx+forecast_len].index, df.iloc[forecast_idx:forecast_idx+forecast_len]['generation(kWh)'])
ax2.set_ylabel('generation(kWh)')
ax2.tick_params(axis='x', rotation=30, labelsize=10, length=0)
ax2.grid(True)

plt.savefig('Plots/Evaluate/error_irradiance.pdf')
plt.show()

## Alternative Loss and Optimisation Functions (GRU)

The following GRU models are not the finalised model, rather studies into different loss and optimisation functions as outlined in the report.

### L2 & Adam

In [ ]:
architecture = 'GRU'
loss_function = 'L2'
optimiser_function = 'Adam'

model_design = f'{architecture}, {loss_function}, {optimiser_function}'
title = f'TRAINING MODEL: {model_design}'
print('='*len(title))
print(title)
print('='*len(title))

model_L2, trainer_L2 = train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta)

### Huber & Adam

In [ ]:
architecture = 'GRU'
loss_function = 'Huber'
optimiser_function = 'Adam'

model_design = f'{architecture}, {loss_function}, {optimiser_function}'
title = f'TRAINING MODEL: {model_design}'
print('='*len(title))
print(title)
print('='*len(title))

model_huber, trainer_huber = train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta)

### L1 & SGD

In [ ]:
architecture = 'GRU'
loss_function = 'Huber'
optimiser_function = 'SGD'

model_design = f'{architecture}, {loss_function}, {optimiser_function}'
title = f'TRAINING MODEL: {model_design}'
print('='*len(title))
print(title)
print('='*len(title))

model_SGD, trainer_SGD = train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta)

### Analysis of Alternative GRU Models

In [ ]:
epochs = np.arange(1, trainer_GRU.num_epochs + 1) #Array of number of epochs over all trained models


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

#Plot the loss for both training and validation data over each model trained so far
ax1.plot(epochs, trainer_GRU.loss_train, 'b-', label='L1 & Adam: Train')
ax1.plot(epochs, trainer_GRU.loss_val, 'b--', label='L1 & Adam: Validation')
ax1.plot(epochs, trainer_L2.loss_train, 'r-', label='L2 & Adam: Train')
ax1.plot(epochs, trainer_L2.loss_val, 'r--', label='L2 & Adam: Validation')
ax1.plot(epochs, trainer_huber.loss_train, 'g-', label='Huber & Adam: Train')
ax1.plot(epochs, trainer_huber.loss_val, 'g--', label='Huber & Adam: Validation')
ax1.plot(epochs, trainer_SGD.loss_train, 'y-', label='L1 & SGD: Train')
ax1.plot(epochs, trainer_SGD.loss_val, 'y--', label='L1 & SGD: Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_yscale('log')
ax1.set_title('Loss: Loss Functions and Optimisation Functions Comparison Results')
ax1.grid(True)
ax1.legend()

#Plot the r2 score for both training and validation data over each model trained so far
ax2.plot(epochs, trainer_GRU.r2_train, 'b-', label='L1 & Adam: Train')
ax2.plot(epochs, trainer_GRU.r2_val, 'b--', label='L1 & Adam: Validation')
ax2.plot(epochs, trainer_L2.r2_train, 'r-', label='L2 & Adam: Train')
ax2.plot(epochs, trainer_L2.r2_val, 'r--', label='L2 & Adam: Validation')
ax2.plot(epochs, trainer_huber.r2_train, 'g-', label='Huber & Adam: Train')
ax2.plot(epochs, trainer_huber.r2_val, 'g--', label='Huber & Adam: Validation')
ax2.plot(epochs, trainer_SGD.r2_train, 'y-', label='L1 & SGD: Train')
ax2.plot(epochs, trainer_SGD.r2_val, 'y--', label='L1 & SGD: Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('R2 Score')
ax2.set_yscale('log')
ax2.set_title('R2 Score: Loss Functions and Optimisation Functions Comparison Results')
ax2.grid(True)

fig.tight_layout()
plt.savefig(f'Plots/Training/loss_optim_comparison.pdf')
plt.show()

## Alternative Model Architecture

The following models are an LSTM and a Simple RNN. They are not the finalised model for this project, rather studies into different model designs as outlined in the report.

### LSTM

#### Train Model

In [ ]:
#TRAIN MODEL
architecture = 'LSTM'
loss_function = 'L1'
optimiser_function = 'Adam'

model_design = f'{architecture}, {loss_function}, {optimiser_function}'
title = f'TRAINING MODEL: {model_design}'
print('='*len(title))
print(title)
print('='*len(title))

model_LSTM, trainer_LSTM = train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta)

#### Evaluate Model

In [ ]:
#Run prediction for entire test dataset
start_date = '2023-09-10 00:00:00'
num_days = 107

forecast_idx = df.index.get_loc(start_date)
forecast_len = int(96 * num_days)

y_trues, y_hats = prediction(model_LSTM, data, scalar_y, forecast_idx, forecast_len)

#Plot Data and Analyse
title = f'Entire Test Dataset - {model_design}'
plot_forecast(y_trues, y_hats, title, forecast_idx, forecast_len)
plot_residuals(y_trues, y_hats, title, forecast_idx, forecast_len)

### Simple RNN

#### Train Model

In [ ]:
#TRAIN MODEL
architecture = 'RNN'
loss_function = 'L1'
optimiser_function = 'Adam'

model_design = f'{architecture}, {loss_function}, {optimiser_function}'
title = f'TRAINING MODEL: {model_design}'
print('='*len(title))
print(title)
print('='*len(title))

model_RNN, trainer_RNN = train_model(architecture, loss_function, optimiser_function, input_size, hidden_size, num_layers, output_size, num_epochs, learning_rate, train_loader, val_loader, dropout_val, patience, delta)

#### Evaluate Model

In [ ]:
#Run prediction for entire test dataset
start_date = '2023-09-10 00:00:00'
num_days = 107

forecast_idx = df.index.get_loc(start_date)
forecast_len = int(96 * num_days)

y_trues, y_hats = prediction(model_RNN, data, scalar_y, forecast_idx, forecast_len)

#Plot Data and Analyse
title = 'Entire Test Dataset'
plot_forecast(y_trues, y_hats, title, forecast_idx, forecast_len)
plot_residuals(y_trues, y_hats, title, forecast_idx, forecast_len)

## Analysis of Alternative Model Architecture

In [ ]:
epochs = np.arange(1, trainer_GRU.num_epochs + 1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

#Loss plots
ax1.plot(epochs, trainer_GRU.loss_train, 'b-', label='GRU: Train')
ax1.plot(epochs, trainer_GRU.loss_val, 'b--', label='GRU: Validation')
ax1.plot(epochs, trainer_LSTM.loss_train, 'r-', label='LSTM: Train')
ax1.plot(epochs, trainer_LSTM.loss_val, 'r--', label='LSTM: Validation')
ax1.plot(epochs, trainer_RNN.loss_train, 'g-', label='Simple RNN: Train')
ax1.plot(epochs, trainer_RNN.loss_val, 'g--', label='Simple RNN: Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_yscale('log')
ax1.set_title('Loss: Model Architecture Comparison Results')
ax1.grid(True)
ax1.legend()

#r2 plots
ax2.plot(epochs, trainer_GRU.r2_train, 'b-', label='GRU: Train')
ax2.plot(epochs, trainer_GRU.r2_val, 'b--', label='GRU: Validation')
ax2.plot(epochs, trainer_LSTM.r2_train, 'r-', label='LSTM: Train')
ax2.plot(epochs, trainer_LSTM.r2_val, 'r--', label='LSTM: Validation')
ax2.plot(epochs, trainer_RNN.r2_train, 'g-', label='Simple RNN: Train')
ax2.plot(epochs, trainer_RNN.r2_val, 'g--', label='Simple RNN: Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('R2 Score')
ax2.set_yscale('log')
ax2.set_title('R2 Score: Model Architecture Comparison Results')
ax2.grid(True)

fig.tight_layout()
plt.savefig(f'Plots/Training/model_architecture_comparison.pdf')
plt.show()

## Feature Data Analysis

### Correlation Matrix

In [ ]:
#Correlation matrix
matrix = df.corr() #Define correlation matrix for full dataframe
plt.figure(figsize=(12,8))
sns.heatmap(matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5) #Plot dataframe matrix
plt.tick_params(axis='x', rotation=45, labelsize=8)
plt.tick_params(axis='y', labelsize=8)
plt.title('Correlation Heatmap')
plt.savefig('Plots/Analysis/corr.pdf')
plt.show()

### PCA

In [ ]:
pca = PCA() #Define pca function object

X_transformed = scalar_X.transform(df[feature_cols]) #Get a transformed object solely for input features

pca.fit(X_transformed) #Determine principal components from critical scaled data

#Bar plot variance
pca.explained_variance_ratio_ #Defines how much explained variance is defined by each feature
plt.figure(figsize=(12,8))
plt.bar(x=range(1,len(pca.explained_variance_ratio_)+1),height=pca.explained_variance_ratio_)
plt.xlabel('Component Index');
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance From Each Princible Component')
plt.xticks(range(1, 15))
plt.grid(True)
plt.savefig('Plots/Analysis/PCA_bar.pdf')


#Cumulative variance
plt.figure(figsize=(12,8))
component_index = np.arange(1,len(pca.explained_variance_ratio_)+1)
plt.plot(component_index, np.cumsum(pca.explained_variance_ratio_), '--o')
plt.xlabel('Component Index');
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance From Each Princible Component')
plt.xticks(range(min(component_index), max(component_index)+1))
plt.grid(True)
plt.savefig('Plots/Analysis/PCA_line.pdf')


#Exact explained variance information for PC1 & PC2 
pc1_var, pc2_var = pca.explained_variance_ratio_[0], pca.explained_variance_ratio_[1]

print(f'PC1 Explained Variance: {pc1_var:.2f}')
print(f'PC2 Explained Variance: {pc2_var:.2f}')

PC1 = pca.components_[0]
PC2 = pca.components_[1]

#Calculate loadings for each variable in PCA analysis
loading_PC1 = PC1 * np.sqrt(pca.explained_variance_[0]) #Defines the contribution of each feature on PC1 explained variance
loading_PC2 = PC2 * np.sqrt(pca.explained_variance_[1]) #Defines the contribution of each feature on PC2 explained variance

df_loadings = pd.DataFrame(np.column_stack([loading_PC1, loading_PC2]),columns=['PC1', 'PC2'], index=feature_cols).round(3)
df_loadings

## Additional Analysis

### Optuna Hyperparameter Tuning

This is commented out in the finalised project due to how long it takes to run.

In [ ]:
def objective(trial):
    early_stopper_trial = EarlyStopping(patience, delta)
    
    hidden_size_trial = trial.suggest_categorical('hidden_size', [64, 128, 256, 512])
    num_layers_trial = trial.suggest_int('num_layers', 1, 3)
    learning_rate_trial = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    model = ModelGRU(loss_function=loss_function, optimiser_function=optimiser_function, input_size=input_size, hidden_size=hidden_size_trial, num_layers=num_layers_trial, output_size=output_size, dropout_val=dropout_val).to(device)
    trainer = Trainer(model=model, num_epochs=num_epochs, learning_rate=learning_rate_trial, train_loader=train_loader, val_loader=val_loader, early_stopper=early_stopper_trial)

    trainer.fit(trial=trial)

    return trainer.loss_val.min()

In [ ]:
#Commented out due to long runtime (~10hrs)

# study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)) #Startup trials is number of trials that must run before pruning and warmup steps is number of steps within trial before it can be pruned

# study.optimize(objective, n_trials=100) 

# title = 'BEST TRIAL RESULTS'
# print('\n' + '='*len(title))
# print(title)
# print('='*len(title))
# print(f'Val Loss: {study.best_value:.4f}')
# print(f'Hyperparamters: {study.best_params}')

# Generalised PV Stations Forecast

The following code represents the data collection and manipulation performed on the remaining PV dataset. This is not completed, however, due to time limitations of the project, and due to the quality of the data being harder to work with than expected. The intended project pipeline that was planned to be achieved is as follows:

Data preparation --> Separate site batches --> Train on entire dataset --> Predict for single PV station (likely SQ567)

For the aims and objectives of this project, this task was deemed less important than effectively evaluating and optimising the single PV site forecasting model constructed above.

## Data Preparation

### PV Metadata Dataset

In [ ]:
#How to use the query library was found from the original datasets corresponding paper
metadata_path = os.path.join(dataset_root, 'Metadata/PV generation system metadata.ttl')
graph = Graph()
graph.parse(metadata_path, format='turtle')

In [ ]:
#Query for metadata dataset to the extraction of several pieces of info for PV sites
query = '''
select ?station ?Latitude_degree ?Longitude_degree ?Altitude_m ?Azimuth ?TiltAngle ?Ratedpower_kW ?Connectiontime ?Contractor where {
    ?station a brick:PV_Generation_System ;
    
    brick:coordinates [ brick:latitude ?Latitude_degree ;
            brick:longitude ?Longitude_degree ] ;
            
    ext:altitude [ brick:hasUnit unit:M ;
            brick:value ?Altitude_m ] ;
            
    ext:azimuth [ brick:value ?Azimuth ] ;
    
    ext:tiltAngle [ brick:value ?TiltAngle ] ;
    
    ext:connectionDate [ brick:value ?Connectiontime ] ;
    
    ext:contractor [ brick:value ?Contractor ] ;
    
    ext:ratedPowerOutput [ brick:hasUnit unit:KW ;
            brick:value ?Ratedpower_kW ] .
}
'''
results = graph.query(query)

In [ ]:
today = datetime.date.today()
today = pd.to_datetime(today)

df_PV_meta = pd.DataFrame(results, columns=results.vars)
df_PV_meta.columns = df_PV_meta.columns.str.strip()
df_PV_meta[['Namespace', 'site']] = df_PV_meta['station'].str.split('#', expand=True)
df_PV_meta['Connectiontime'] = pd.to_datetime(df_PV_meta['Connectiontime'])

PV_meta_cols = ['site', 'Latitude_degree', 'Altitude_m', 'Ratedpower_kW']
df_PV_meta = df_PV_meta[PV_meta_cols].set_index('site')

df_PV_meta

### PV Generation Datasets

In [ ]:
def PV_files_merge(folder_path):

    df = None
    
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv') or f.endswith('.xlsx')]

    for file in files:
        file_path = os.path.join(folder_path, file) #Find exact file path
        
        if file.endswith('.csv'):
            df_file = pd.read_csv(file_path) #Read csv if ends as such
        elif file.endswith('.xlsx'):
            df_file = pd.read_excel(file_path) #Read excel file is ends as such

        df_file['Time'] = pd.to_datetime(df_file['Time'], format='mixed')
        df_file = df_file.set_index('Time').sort_index()

        df_file = df_file[~df_file.index.duplicated(keep='first')] #Drops columns where duplicated is equal to false

        df_file = df_file.resample('60min').mean().interpolate() #Resample time step of generation to 60min so that all are equal
        
        #Change gen and power column names
        site_name = file.replace(' ', '_').replace('.csv', '').replace('.xlsx', '')

        df_file['site'] = site_name

        if df is None:
            df = df_file
        else:
            df = pd.concat([df, df_file], axis=0).sort_index()
            
    df = df.sort_index()
    
    return df

In [ ]:
yes_optim_root = os.path.join(dataset_root, 'Time series dataset/PV generation dataset/PV stations with panel level optimizer/Site level dataset')
no_optim_root = os.path.join(dataset_root, 'Time series dataset/PV generation dataset/PV stations without panel level optimizer/Site level dataset')

df_yes_optim = PV_files_merge(yes_optim_root)    
df_no_optim = PV_files_merge(no_optim_root)

df_PV_gen = pd.concat([df_yes_optim, df_no_optim], axis=0)
df_PV_gen = df_PV_gen.sort_index()

#Merge PV data with metadata
# df_PV_gen = df_PV_gen.join(df_PV_meta, on='site')

df_PV_gen

In [ ]:
#AI Code / ChatGPT / 'Im attempting to plot two columsn within my dataframe for energy and power generation over 60 sites where one site equals 1 row on the plot. I want to plot each site per row for a clear dist of each site data. How should i use the dataframe to achieve this?'

PV_sites = df_PV_gen['site'].unique()
n_sites = len(PV_sites)

min_time = df_PV_gen.index.min()
max_time = df_PV_gen.index.max()

fig, axes = plt.subplots(nrows=n_sites, ncols=2, figsize=(12, n_sites * 2))

for i, site in enumerate(PV_sites):

    site_df = df_PV_gen[df_PV_gen['site'] == site]

    #Energy generation plot
    sns.lineplot(data=site_df, x=site_df.index, y='generation(kWh)', ax=axes[i, 0])

    axes[i, 0].set_xlim(min_time, max_time)
    axes[i, 0].set_title(f'{site} — generation(kWh)')
    axes[i, 0].grid(True)

    #Power plot
    sns.lineplot(data=site_df, x=site_df.index, y='power(W)', ax=axes[i, 1])

    axes[i, 1].set_xlim(min_time, max_time)
    axes[i, 1].set_title(f'{site} — power(W)')
    axes[i, 1].grid(True)

#Rotate x labels
for ax in axes.flatten():
    ax.tick_params(axis='x', rotation=30, labelsize=10, length=0)

plt.tight_layout()

plt.savefig('Plots/Generalised/PV_energy_power_per_site.pdf')
plt.show()